# Cleanups
I discovered those late (after generation of the torch file for *clews*). 
However, these cleanups are necessary. 

### Requirements

`python collect_metadata.py --audio-dir /data/sha/audio/mp4/ --dvi-file ../discogs-vi-2/data/dvi2/dataset/divers1m/dvi_fm.jsonl --meta-file ../clews/cache/metadata-dvi2fm.pt --njobs 96`

Afterwards (might need debug):

`python enrich_cache_metadata.py --input ../clews/cache/metadata-dvi2fm.pt --output metadata-dvi2fm_aux.pt`


In [5]:
import os
import torch
import pandas as pd

def load_dataset(path):
    """
    Load dataset from the specified path.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dataset not found at {path}")
    
    def inverse_split_dict(split_dict):
        # Add split column to dvi2Torch based on dvi2_torch["split"]
        clique_to_split = {}
        for split_name, sub_dict in split_dict.items():
            for clique in sub_dict.keys():
                clique_to_split[clique] = split_name
        return clique_to_split

    meta = torch.load(path, weights_only=False)
    if isinstance(meta, dict) and "info" in meta:
        info =  meta["info"]
        split = meta["split"]
    else:
        info, split = meta

    df = pd.DataFrame.from_dict(info, orient="index")
           
    clique2split = inverse_split_dict(split)
    df["split"] = df["clique"].map(clique2split)
    
    if "youtube_id" not in df.columns:
        df["youtube_id"] = df.filename.apply(lambda x: x.split("/")[-1].split(".")[0])
    
    df["dvi"] = ~df.apply(lambda x: x.youtube_id in x.version, axis=1)
    
    return df, meta

df_divers_aux, meta_divers_aux = load_dataset("../vi-mtl/cache/metadata-dvi2fm_aux.pt")


In [6]:
# deduplicate the dataframes
def deduplicate_youtube_versions(df):
    # Identify duplicated youtube_ids (more than once)
    duplicated_ids = df['youtube_id'][df['youtube_id'].duplicated(keep=False)]

    # Filter rows where youtube_id is duplicated and youtube_id == version
    to_drop = df[
        df['youtube_id'].isin(duplicated_ids) &
        (df['youtube_id'] == df['version'])
    ]

    # Drop those rows
    df_cleaned = df.drop(to_drop.index)

    return df_cleaned

print(f"Duplicated versions: {df_divers_aux.youtube_id.duplicated().sum():,}")
df_divers_aux2 = deduplicate_youtube_versions(df_divers_aux)
print(f"Duplicated versions: {df_divers_aux2.youtube_id.duplicated().sum():,}")



Duplicated versions: 126,487
Duplicated versions: 51


In [7]:
def filter_non_singleton_cliques(df):
    """
    Filter out cliques that have only one entry.
    """
    clique_counts = df.groupby("clique").size()
    non_singleton_cliques = clique_counts[clique_counts > 1].index
    return df[df["clique"].isin(non_singleton_cliques)]

df_divers_aux2 = filter_non_singleton_cliques(df_divers_aux2)


In [8]:
def deduplicate_false_cliques(df):
    """here we remove false clique splits where multiple cliques have the same youtube_id"""
    # Result holder for cleaned rows
    keep_rows = []
    grouped = df.groupby('youtube_id', sort=False)

    for youtube_id, group in grouped:
        if len(group) == 1:
            keep_rows.append(group)
        else:
            clique_counts = group['clique'].value_counts()
            max_count = clique_counts.max()
            top_cliques = clique_counts[clique_counts == max_count].index
            first_clique = group[group['clique'].isin(top_cliques)].iloc[0]['clique']
            filtered = group[group['clique'] == first_clique]
            keep_rows.append(filtered)

    # Concatenate all kept groups
    df_cleaned = pd.concat(keep_rows).reset_index(drop=True)

    return df_cleaned

df_divers_aux3 = deduplicate_false_cliques(df_divers_aux2)
df_divers_aux3 = filter_non_singleton_cliques(df_divers_aux3)
print(f"Unique YouTube IDs: {df_divers_aux3.youtube_id.nunique():,}, df length: {len(df_divers_aux3):,}")

df_divers_aux3["index"] = df_divers_aux3.apply(lambda x: f"{x['clique']}:{x['version']}", axis=1)
df_divers_aux3 = df_divers_aux3.set_index("index", drop=False)


Unique YouTube IDs: 1,244,810, df length: 1,244,810


In [9]:
def print_df_summary(df, subset="overall"):
    if subset != "overall":
        df = df.query(f"split == '{subset}'")
    print(f"Summary statistics for {subset}")

    # Column stats
    stats = []
    for col in ["clique", "version", "youtube_id"]:
        unique_count = df[col].nunique()
        total_count = df[col].count()
        stats.append(f"{col:<12} | Unique: {unique_count:>7,} | Total: {total_count:>7,}")
    print("\n".join(stats))

    # Version per clique stats
    version_per_clique = df.groupby("clique")["version"].nunique()
    print("\nVersion per clique:")
    print(f"  Min: {version_per_clique.min():>3}  | Max: {version_per_clique.max():>3}  | "
          f"Mean: {version_per_clique.mean():>5.2f}  | Median: {version_per_clique.median():>5.2f}  | "
          f"Std: {version_per_clique.std():>5.2f}")
    print("=" * 40)
    

# Stats

In [13]:
print_df_summary(df_divers_aux3, "train")
print_df_summary(df_divers_aux3, "valid")
print_df_summary(df_divers_aux3, "test")


Summary statistics for train
clique       | Unique:  75,074 | Total: 935,481
version      | Unique: 935,481 | Total: 935,481
youtube_id   | Unique: 935,481 | Total: 935,481

Version per clique:
  Min:   2  | Max: 472  | Mean: 12.46  | Median:  5.00  | Std: 18.23
Summary statistics for valid
clique       | Unique:   8,344 | Total: 102,350
version      | Unique: 102,350 | Total: 102,350
youtube_id   | Unique: 102,350 | Total: 102,350

Version per clique:
  Min:   2  | Max: 249  | Mean: 12.27  | Median:  4.00  | Std: 17.87
Summary statistics for test
clique       | Unique:  10,346 | Total: 206,979
version      | Unique: 206,979 | Total: 206,979
youtube_id   | Unique: 206,979 | Total: 206,979

Version per clique:
  Min:   2  | Max: 612  | Mean: 20.01  | Median:  6.00  | Std: 32.47


## DVI2

In [15]:
dvi2 = df_divers_aux3[df_divers_aux3["dvi"]]
dvi2 = filter_non_singleton_cliques(dvi2)
print_df_summary(dvi2, "train")
print_df_summary(dvi2, "valid")
print_df_summary(dvi2, "test")


Summary statistics for train
clique       | Unique:  71,783 | Total: 300,053
version      | Unique: 300,053 | Total: 300,053
youtube_id   | Unique: 300,053 | Total: 300,053

Version per clique:
  Min:   2  | Max: 445  | Mean:  4.18  | Median:  2.00  | Std:  8.33
Summary statistics for valid
clique       | Unique:   7,976 | Total:  32,780
version      | Unique:  32,780 | Total:  32,780
youtube_id   | Unique:  32,780 | Total:  32,780

Version per clique:
  Min:   2  | Max: 249  | Mean:  4.11  | Median:  2.00  | Std:  7.93
Summary statistics for test
clique       | Unique:  10,052 | Total: 108,153
version      | Unique: 108,153 | Total: 108,153
youtube_id   | Unique: 108,153 | Total: 108,153

Version per clique:
  Min:   2  | Max: 612  | Mean: 10.76  | Median:  3.00  | Std: 27.44


## YVI

In [16]:
df_yvi, meta_yvi = load_dataset("cache/metadata-yvi.pt")


In [19]:
df_yvi[df_yvi.dvi]

,id,clique,version,artist,title,filename,samplerate,length,channels,split,youtube_id,dvi
C-0000023_0:V-0000182,12,C-0000023_0,V-0000182,Harold Burrage,Got To Find A Way - Original,xP/xPxaDKl8R6g.mp4,44100,220.971247,2,train,xPxaDKl8R6g,True
C-0000062_0:V-0000872,112,C-0000062_0,V-0000872,Yell!,Instant Replay (Extended),Rh/Rh_1YmbqjSI.mp4,44100,356.297143,2,train,Rh_1YmbqjSI,True
C-0000116_1:V-0001214,175,C-0000116_1,V-0001214,Halsey,Whispers,dK/dK3OJ1grBg4.mp4,44100,191.714104,2,train,dK3OJ1grBg4,True
C-0000145_0:V-0001536,219,C-0000145_0,V-0001536,?,Nicole Scherzinger - Right There,Yk/YkdkX6_3iGI.mp4,44100,218.639093,2,train,YkdkX6_3iGI,True
C-0000223_0:V-0001960,271,C-0000223_0,V-0001960,Venom,Bloodlust (Radio 1 Session),BP/BPnf2YM5Wf0.mp4,44100,155.816054,2,train,BPnf2YM5Wf0,True
...,...,...,...,...,...,...,...,...,...,...,...,...
C-0347411_0:V-1908710,1376283,C-0347411_0,V-1908710,CTMF,Failure Not Success,z0/z0vjdP8TmLc.mp4,44100,186.977234,2,test,z0vjdP8TmLc,True
C-0347542_0:V-1908984,1376288,C-0347542_0,V-1908984,"Wild Billy Childish, The Singing Loins",Stood Upon a Chair,Gw/Gwew4pFwSeg.mp4,44100,170.839365,2,test,Gwew4pFwSeg,True
C-0347644_0:V-1909197,1376293,C-0347644_0,V-1909197,"Gabriel Selvage, Lucio Yanel, Lucio Yanel",La Cara de Pedro,8A/8ATmSb6JsQs.mp4,44100,240.522449,2,test,8ATmSb6JsQs,True
C-0347876_0:V-1909687,1376299,C-0347876_0,V-1909687,Banda Machos,A prueba de balas,Fi/FivTgRSZGnM.mp4,44100,201.675465,2,test,FivTgRSZGnM,True


In [ ]:
yvi_only = df_divers_aux3[~df_divers_aux3["dvi"]]
print_df_summary(yvi_only, "train")
print_df_summary(yvi_only, "valid")
print_df_summary(yvi_only, "test")


Summary statistics for train
clique       | Unique:  42,155 | Total: 632,239
version      | Unique: 632,239 | Total: 632,239
youtube_id   | Unique: 632,239 | Total: 632,239

Version per clique:
  Min:   1  | Max:  97  | Mean: 15.00  | Median:  7.00  | Std: 18.72
Summary statistics for valid
clique       | Unique:   4,615 | Total:  69,211
version      | Unique:  69,211 | Total:  69,211
youtube_id   | Unique:  69,211 | Total:  69,211

Version per clique:
  Min:   1  | Max:  92  | Mean: 15.00  | Median:  7.00  | Std: 18.67
Summary statistics for test
clique       | Unique:   5,387 | Total:  98,539
version      | Unique:  98,539 | Total:  98,539
youtube_id   | Unique:  98,539 | Total:  98,539

Version per clique:
  Min:   1  | Max:  91  | Mean: 18.29  | Median:  9.00  | Std: 20.16


In [35]:
import pandas as pd

# Step 1: Get all non-dvi items
yvi_only = df_divers_aux3[~df_divers_aux3["dvi"]].copy()
cliques_to_consider = yvi_only["clique"].unique()

dvi_candidates = df_divers_aux3[
    (df_divers_aux3["dvi"]) & 
    (df_divers_aux3["clique"].isin(cliques_to_consider))
]

dvi_one_per_clique = (
    dvi_candidates
    .groupby("clique", group_keys=False)
    .apply(lambda x: x.sample(n=1, random_state=42))
)

yvi = pd.concat([yvi_only, dvi_one_per_clique], ignore_index=True)

print_df_summary(yvi, "train")
print_df_summary(yvi, "valid")
print_df_summary(yvi, "test")


/tmp/ipykernel_30554/3147847669.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=1, random_state=42))


Summary statistics for train
clique       | Unique:  42,155 | Total: 674,292
version      | Unique: 674,292 | Total: 674,292
youtube_id   | Unique: 674,292 | Total: 674,292

Version per clique:
  Min:   2  | Max:  98  | Mean: 16.00  | Median:  8.00  | Std: 18.72
Summary statistics for valid
clique       | Unique:   4,615 | Total:  73,817
version      | Unique:  73,817 | Total:  73,817
youtube_id   | Unique:  73,817 | Total:  73,817

Version per clique:
  Min:   2  | Max:  93  | Mean: 16.00  | Median:  8.00  | Std: 18.67
Summary statistics for test
clique       | Unique:   5,387 | Total: 103,919
version      | Unique: 103,919 | Total: 103,919
youtube_id   | Unique: 103,919 | Total: 103,919

Version per clique:
  Min:   2  | Max:  92  | Mean: 19.29  | Median: 10.00  | Std: 20.16


# Check if we have audio file

In [63]:
audio_path = "../../data/audio/mp4/"
exists_series = df_divers_aux3["filename"].apply(lambda fn: os.path.exists(os.path.join(audio_path, fn)))


In [65]:
exists_series.all()

np.True_

# Save

In [59]:
def save_metadata(df, path):
    """
    Save the DataFrame to a specified path.
    """
    df.to_csv(path, index=False)
    print(f"Metadata saved to {path}")

meta_divers_aux["info"]

{'C-0000000_0:V-0000004': {'id': 0,
  'clique': 'C-0000000_0',
  'version': 'V-0000004',
  'artist': 'The Vulgar Boatmen',
  'title': 'Heartbeat',
  'filename': 'MI/MI23duPUNaY.mp4',
  'samplerate': 44100,
  'length': 216.3736961451,
  'channels': 2,
  'youtube_id': 'MI23duPUNaY',
  'track_writer_names': ['Dale Lawrence', 'Robert Ray'],
  'release_artist_names': ['The Vulgar Boatmen'],
  'release_genres': ['Rock'],
  'release_styles': [['Rock', 'Folk Rock']],
  'country': 'Europe',
  'labels': ['EastWest', 'Blanco Y Negro'],
  'formats': ['CD'],
  'released': '1995',
  'yt_title': 'Heartbeat',
  'yt_description': 'Provided to YouTube by TuneCore\n\nHeartbeat · The Vulgar Boatmen\n\nOpposite Sex\n\n℗ 2014 No Nostalgia\n\nReleased on: 2010-08-27\n\nAuto-generated by YouTube.',
  'yt_tags': ['The', 'Vulgar', 'Boatmen', 'Opposite', 'Sex', 'Heartbeat'],
  'yt_categories': ['Music'],
  'yt_channel': 'The Vulgar Boatmen - Topic',
  'yt_upload_date': 20150119.0,
  'yt_view_count': 586.0,
  'te

In [ ]:
output_dir = "cache/divers1m/"

drop = {
       "full": ["dvi", "split"],
       "rich": ["dvi", "split", "yt_description"], 
       "light": 
              ['track_writer_names',
              'release_artist_names', 'release_genres', 'release_styles', 'country',
              'labels', 'formats', 'released', 'yt_title', 'yt_description',
              'yt_tags', 'yt_categories', 'yt_channel', 'yt_upload_date',
              'yt_view_count', 'tempo', 'matched_instruments_groups',
              'matched_concepts', 'matched_release_styles', 'matched_segments',
              'split', 'dvi', 'index']
}


subdirs = ["full", "rich", "light"]

# datasets
# dvi2 (use the df dvi2)
# divers1m (use the df df_divers_aux3)
# yvi (use the df yvi)

df_divers_aux3.to_dict(orient="index")

meta_divers_aux["split"]


{'train': {'C-0000000_0': ['C-0000000_0:V-0000004', 'C-0000000_0:V-0000006'],
  'C-0000006_0': ['C-0000006_0:V-0000103',
   'C-0000006_0:V-0000105',
   'C-0000006_0:V-0000107',
   'C-0000006_0:V-0000109',
   'C-0000006_0:V-0000112'],
  'C-0000015_0': ['C-0000015_0:V-0000155', 'C-0000015_0:V-0000156'],
  'C-0000016_0': ['C-0000016_0:V-0000158', 'C-0000016_0:V-0000159'],
  'C-0000023_0': ['C-0000023_0:V-0000181',
   'C-0000023_0:V-0000182',
   'C-0000023_0:V-0000183',
   'C-0000023_0:V-0000186',
   'C-0000023_0:Bzq5IBa4A9U',
   'C-0000023_0:PBHr_3aNgOg',
   'C-0000023_0:dIwUqjFa5FU',
   'C-0000023_0:xPxaDKl8R6g'],
  'C-0000029_0': ['C-0000029_0:V-0000216',
   'C-0000029_0:V-0000218',
   'C-0000029_0:V-0000219'],
  'C-0000030_0': ['C-0000030_0:V-0000222',
   'C-0000030_0:V-0000227',
   'C-0000030_0:V-0000228',
   'C-0000030_0:V-0000268',
   'C-0000030_0:V-0000271',
   'C-0000030_0:V-0000274',
   'C-0000030_0:V-0000275',
   'C-0000030_0:V-0000276',
   'C-0000030_0:V-0000280',
   'C-0000030

In [53]:
yvi["index"] = yvi.apply(lambda x: f"{x['clique']}:{x['version']}", axis=1)
yvi = yvi.set_index("index", drop=False)

In [54]:
import os
import json
import torch
from tqdm import tqdm
import numpy as np

output_dir = "cache/divers1m/"

drop = {
    "full": ["dvi", "split"],
    "rich": ["dvi", "split", "yt_description"],
    "light": [
        'track_writer_names', 'release_artist_names', 'release_genres', 'release_styles', 'country',
        'labels', 'formats', 'released', 'yt_title', 'yt_description',
        'yt_tags', 'yt_categories', 'yt_channel', 'yt_upload_date',
        'yt_view_count', 'tempo', 'matched_instruments_groups',
        'matched_concepts', 'matched_release_styles', 'matched_segments',
        'split', 'dvi', 'index'
    ]
}

subdirs = ["full", "rich", "light"]

datasets = {
    "dvi2": dvi2,
    "divers1m": df_divers_aux3,
    "yvi": yvi
}

os.makedirs(output_dir, exist_ok=True)

def convert_ndarrays_to_lists(d):
    for k, subdict in d.items():
        for subk, v in subdict.items():
            if isinstance(v, np.ndarray):
                subdict[subk] = v.tolist()
    return d

for name, df in datasets.items():
    print(f"\nProcessing dataset: {name}")

    # Build split dict from df once
    print("    > Building split dict from DataFrame...")
    split_dict = {}
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            clique, version = idx.split(":")
        except ValueError:
            raise ValueError(f"Invalid index format: '{idx}' (expected 'clique:version')")
        split = row["split"]
        split_dict.setdefault(split, {}).setdefault(clique, []).append(version)

    for subdir in subdirs:
        print(f"  - Subdir: {subdir}")
        subpath = os.path.join(output_dir, subdir)
        os.makedirs(subpath, exist_ok=True)

        # Drop columns
        df_filtered = df.drop(columns=[col for col in drop[subdir] if col in df.columns])

        # Build info dict
        info_dict = convert_ndarrays_to_lists(df_filtered.to_dict(orient="index"))

        # Build metadata dict
        metadata = {
            "info": info_dict,
            "split": split_dict
        }

        # Save JSON
        json_path = os.path.join(subpath, f"{name}.json")
        with open(json_path, "w") as f:
            json.dump(metadata, f, indent=2)
        print(f"    ✔ Saved JSON: {json_path}")

        # Save Torch
        torch_path = os.path.join(subpath, f"{name}.pt")
        torch.save(metadata, torch_path)
        print(f"    ✔ Saved Torch: {torch_path}")



Processing dataset: yvi
    > Building split dict from DataFrame...


  0%|          | 0/852028 [00:00<?, ?it/s]

100%|██████████| 852028/852028 [00:32<00:00, 26010.98it/s]


  - Subdir: full
    ✔ Saved JSON: cache/divers1m/full/yvi.json
    ✔ Saved Torch: cache/divers1m/full/yvi.pt
  - Subdir: rich
    ✔ Saved JSON: cache/divers1m/rich/yvi.json
    ✔ Saved Torch: cache/divers1m/rich/yvi.pt
  - Subdir: light
    ✔ Saved JSON: cache/divers1m/light/yvi.json
    ✔ Saved Torch: cache/divers1m/light/yvi.pt
